# 🌊 AquaEnhance — Underwater Image Enhancement

**Google Colab Training Notebook**  
Trains a U-Net on the UIEB dataset to enhance degraded underwater images.

---
**Before running:**
1. `Runtime → Change runtime type → T4 GPU`
2. Upload `kaggle.json` when Cell 2 prompts you
3. Run cells top-to-bottom — Google Drive keeps the model after session ends

In [ ]:
# ── Cell 1 : Install dependencies ───────────────────────────────────
!pip install -q kaggle scikit-image opencv-python-headless Pillow
import tensorflow as tf
print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ── Cell 2 : Configure Kaggle API key ───────────────────────────────
# 1. Go to https://www.kaggle.com → Account → API → Create New Token
# 2. Upload the kaggle.json file when the dialog appears below
import os
from google.colab import files

uploaded = files.upload()          # select your kaggle.json
os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json
print('Kaggle API key ready.')

In [ ]:
# ── Cell 3 : Mount Google Drive (persistent model storage) ──────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/AquaEnhance'
MODEL_DIR = DRIVE_DIR + '/model'
DATA_DIR  = DRIVE_DIR + '/data'
for d in [MODEL_DIR, DATA_DIR]:
    os.makedirs(d, exist_ok=True)
print('Drive mounted. Model will be saved to:', MODEL_DIR)

In [ ]:
# ── Cell 4 : Upload project files ────────────────────────────────────
# OPTION A — if you have the project on GitHub:
#   !git clone https://github.com/YOUR_USERNAME/AquaEnhance.git /content/AquaEnhance
#   %cd /content/AquaEnhance
#
# OPTION B — manual upload (zip the ml/ folder, upload here):
#   from google.colab import files
#   files.upload()          # upload ml.zip
#   !unzip -q ml.zip -d /content/

# After either option, add ml/ to Python path:
import sys
sys.path.insert(0, '/content/ml')     # adjust if cloned elsewhere
print('sys.path set. Ready to import project modules.')

In [ ]:
# ── Cell 5 : Download UIEB dataset ───────────────────────────────────
import subprocess, os
result = subprocess.run(
    ['kaggle', 'datasets', 'download',
     '-d', 'larjeck/uieb-dataset-raw-and-reference',
     '-p', DATA_DIR, '--unzip'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

raw_n = len(os.listdir(DATA_DIR + '/raw-890'))
ref_n = len(os.listdir(DATA_DIR + '/reference-890'))
print(f'raw-890: {raw_n} files | reference-890: {ref_n} files')

In [ ]:
# ── Cell 6 : Build train / val / test datasets ───────────────────────
from pathlib import Path
import dataset as ds

# Point the module to the Drive data directory
ds.RAW_DIR = Path(DATA_DIR) / 'raw-890'
ds.REF_DIR = Path(DATA_DIR) / 'reference-890'

pairs = ds.get_pairs(ds.RAW_DIR, ds.REF_DIR)
train_pairs, val_pairs, test_pairs = ds.split_pairs(pairs)

BATCH    = 8
train_ds = ds.build_dataset(train_pairs, batch_size=BATCH, shuffle=True)
val_ds   = ds.build_dataset(val_pairs,   batch_size=BATCH)
test_ds  = ds.build_dataset(test_pairs,  batch_size=BATCH)
print('Datasets built.')

In [ ]:
# ── Cell 7 : Build U-Net model ────────────────────────────────────────
from unet import build_unet
import tensorflow as tf

model = build_unet()
model.summary(line_length=90)
print(f'Total parameters: {model.count_params():,}')

In [ ]:
# ── Cell 8 : Define loss and compile ─────────────────────────────────
@tf.keras.saving.register_keras_serializable(package='AquaEnhance')
def combined_loss(y_true, y_pred):
    mae  = tf.reduce_mean(tf.abs(y_true - y_pred))
    ssim = tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))
    return 0.5 * mae + 0.5 * (1.0 - ssim)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=combined_loss,
    metrics=['mae'],
)
print('Model compiled.')

In [ ]:
# ── Cell 9 : Train ────────────────────────────────────────────────────
BEST_PATH = MODEL_DIR + '/best_model.keras'

class CheckpointEveryN(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % 5 == 0:
            p = MODEL_DIR + f'/ckpt_epoch_{epoch+1:03d}.keras'
            self.model.save(p)
            print(f'  Checkpoint → {p}')

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        BEST_PATH, monitor='val_loss', save_best_only=True, verbose=1),
    CheckpointEveryN(),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=10,
        restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=5, min_lr=1e-7, verbose=1),
]

history = model.fit(
    train_ds, validation_data=val_ds,
    epochs=100, callbacks=callbacks,
)
print('Training complete. Best model at:', BEST_PATH)

In [ ]:
# ── Cell 10 : Plot training curves ───────────────────────────────────
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['loss'],     label='train')
ax1.plot(history.history['val_loss'], label='val')
ax1.set(title='Loss (combined)', xlabel='Epoch'); ax1.legend()

ax2.plot(history.history['mae'],     label='train')
ax2.plot(history.history['val_mae'], label='val')
ax2.set(title='MAE', xlabel='Epoch'); ax2.legend()

plt.tight_layout()
plt.savefig(MODEL_DIR + '/training_curves.png', dpi=120)
plt.show()

In [ ]:
# ── Cell 11 : Evaluate on test set ───────────────────────────────────
import numpy as np
from PIL import Image
import evaluate as ev
from inference import enhance

all_metrics = []
for raw_path, ref_path in test_pairs:
    enhanced = enhance(raw_path, BEST_PATH)
    ref      = np.array(Image.open(ref_path).convert('RGB'),
                        dtype=np.float32) / 255.0
    all_metrics.append(ev.compute_all_metrics(enhanced, reference=ref))

print('\nTest set averages')
print('-' * 30)
for key in all_metrics[0]:
    vals = [m[key] for m in all_metrics]
    print(f'  {key:8s}: {np.mean(vals):.4f}')

In [ ]:
# ── Cell 12 : Visual comparison (first 3 test images) ───────────────
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

fig, axes = plt.subplots(3, 2, figsize=(10, 12))
for i, (raw_path, _) in enumerate(test_pairs[:3]):
    raw = np.array(Image.open(raw_path).convert('RGB'),
                   dtype=np.float32) / 255.0
    enh = enhance(raw_path, BEST_PATH)
    axes[i, 0].imshow(raw); axes[i, 0].set_title('Raw (Degraded)')
    axes[i, 0].axis('off')
    axes[i, 1].imshow(enh); axes[i, 1].set_title('Enhanced (U-Net)')
    axes[i, 1].axis('off')

plt.tight_layout()
plt.savefig(MODEL_DIR + '/sample_results.png', dpi=150)
plt.show()
print('Saved to', MODEL_DIR + '/sample_results.png')